# 04 · Training Monitoring & Failure Diagnosis


1. The Training Doctor
This notebook analyses the behaviour of the models trained in the project.

Monitoring tells us what happened during training. Failure diagnosis explains why it happened and how to fix it.

2. Diagnostic Framework
Healthy Learning
Symptoms:

Training loss decreases.
Validation loss decreases.
Training accuracy increases.
Validation accuracy increases.
Diagnosis: The model is learning useful patterns and generalizing to unseen data.

Overfitting
Symptoms:

Training loss decreases.
Validation loss increases.
Training accuracy becomes much larger than validation accuracy.
Diagnosis: The model memorizes the training set instead of learning general features.

Possible fixes:

Weight decay (L2 regularization)
Dropout
Early stopping
More training data
Underfitting
Symptoms:

Training accuracy remains low.
Validation accuracy remains low.
Diagnosis: The model is too simple or has not been trained enough.

Possible fixes:

Larger model
More training epochs
Better feature extraction
Random Guessing
Symptoms:

Loss remains close to ln(C), where C is the number of classes.
Accuracy remains close to chance level.
Diagnosis: The model is not learning meaningful patterns.

Possible causes:

Incorrect labels
Bug in the training loop
Learning rate issues
Unstable Training
Symptoms:

Loss oscillates heavily.
Loss becomes NaN.
Accuracy fluctuates dramatically.
Diagnosis: Training is unstable.

Possible fixes:

Lower learning rate
Gradient clipping
Better initialization


3. Baseline Model Diagnosis
Evidence of Overfitting
The logistic regression baseline provides a clear example of overfitting.

When the regularization parameter was set to C = 0.1 (weak regularization), the model achieved:

Training accuracy: 99.8%
Validation accuracy: 20.4%
This large gap between training and validation performance indicates that the model memorized the training images instead of learning patterns that generalize to unseen artworks.

Impact of Regularization
Increasing the regularization strength significantly reduced overfitting.

With C = 0.0001:

Training accuracy: 44.6%
Validation accuracy: 26.8%
Although training accuracy decreased, validation performance improved and the train-validation gap became much smaller.

This demonstrates one of the key lessons from Session 6:

A model should not be evaluated by its training accuracy alone. The most important metric is its performance on unseen data.

Diagnosis
Diagnosis: Severe overfitting with weak regularization.

Treatment:

Stronger L2 regularization
Early stopping
More training data
More informative image representations


In [1]:
# Load the final evaluation results produced by the project.
# These JSON files contain the metrics of the selected ResNet models
# after training and testing.

import json

# Load genre classification results
with open("../results/resnet.json") as f:
    genre_results = json.load(f)

# Load style classification results
with open("../results/resnet_style.json") as f:
    style_results = json.load(f)

# Display the final metrics used in the diagnosis section
print("Genre classification results:")
print(genre_results)

print("\nStyle classification results:")
print(style_results)

Genre classification results:
{'model': 'resnet18_finetuned_lr1e-4', 'test_accuracy': 0.5859, 'cpu_ms_per_image': 15.51, 'n_parameters': 11182155, 'size_mb': 44.7, 'train_time_s': 98.1, 'n_classes': 11, 'img_size': 128, 'val_acc_head_only': 0.5706, 'val_acc_finetuned_lr1e-4': 0.5847, 'val_acc_finetuned_lr1e-5': 0.5645}

Style classification results:
{'model': 'resnet18_style_finetuned_lr1e-4', 'label': 'style', 'test_accuracy': 0.3658, 'cpu_ms_per_image': 13.23, 'n_parameters': 11190363, 'size_mb': 44.8, 'train_time_s': 212.2, 'n_classes': 27, 'img_size': 128, 'val_acc_head_only': 0.3533, 'val_acc_finetuned_lr1e-4': 0.3792, 'val_acc_finetuned_lr1e-5': 0.3631}


4. ResNet Diagnosis (Genre Classification)

Observed Training Behaviour
The ResNet model achieved approximately 58.6% test accuracy on genre classification.

The training curves show a clear divergence between training and validation performance:

Training accuracy increases from approximately 61% to nearly 99%.
Validation accuracy remains close to 57-58%.
Training loss decreases continuously.
Validation loss increases after the first epochs.
Diagnosis
The model is strongly overfitting.

The network continues improving on the training set while its performance on unseen data remains almost constant. This indicates that the model is memorizing training examples faster than it is learning general artistic features.

Evidence
At the end of training:

Training accuracy ≈ 99%
Validation accuracy ≈ 57%
The large generalization gap suggests that additional epochs mainly improve memorization rather than generalization.

Role of Early Stopping
The best validation performance is reached very early during training.

Early stopping prevents selecting a model from the final epochs, where validation loss becomes substantially larger despite near-perfect training accuracy.

Conclusion
Despite the observed overfitting, the ResNet model significantly outperforms the logistic regression baseline and provides the best overall performance for genre classification.

5. ResNet Diagnosis (Style Classification)

Observed Training Behaviour
The ResNet model achieved approximately 36.6% test accuracy on style classification.

The training curves show a behaviour similar to the genre classification task:

Training accuracy increases from approximately 45% to more than 95%.
Validation accuracy remains close to 36-38%.
Training loss continues decreasing.
Validation loss increases after the first epochs.
Diagnosis
The model exhibits strong overfitting.

While the network nearly memorizes the training images, validation performance remains nearly constant throughout the training process.

This indicates that style classification is a particularly difficult task, requiring the model to distinguish artistic techniques rather than simply identify visual content.

Evidence
At the end of training:

Training accuracy ≈ 96%
Validation accuracy ≈ 37%
The large gap between training and validation performance suggests limited generalization.

Why Style Classification Is Harder
Genre classification focuses on the content of the painting, such as landscapes, portraits, or cityscapes.

Style classification requires identifying artistic characteristics such as brushwork, colour palettes, composition, and painting techniques.

These patterns are more subtle and often overlap across artistic movements.

Conclusion
Although style classification remains challenging, fine-tuned ResNet models still achieve substantially better results than classical approaches and provide useful predictions for a museum guide application.

6. Error Analysis and Confusion Matrices

Genre Classification Errors

The confusion matrix reveals that some genre categories are visually easier to distinguish than others. Most mistakes occur between categories that share similar visual content and composition.

Landscape vs Cityscape
Landscape paintings are frequently confused with cityscapes and vice versa. Both categories often contain outdoor scenes, horizons, and similar spatial organization, making the distinction difficult for the model.

Genre Painting vs Religious Painting
Genre paintings are often confused with religious paintings. Many artworks in both categories contain multiple human figures arranged in narrative scenes, making the distinction difficult from visual information alone.

Nude Painting vs Religious Painting
The model sometimes confuses nude paintings with religious paintings. Both categories frequently contain human bodies as dominant visual elements and may share similar colour distributions and compositions.

Implications
The errors are generally understandable and occur between visually related categories rather than completely unrelated ones.

This suggests that the model learned meaningful artistic features even when it fails to predict the correct label.

Style Classification Errors

Style classification is substantially more difficult than genre classification. The confusion matrix shows that the model struggles most with artistic movements that share similar visual characteristics.

Impressionism vs Post-Impressionism
The model frequently confuses Impressionism and Post-Impressionism. This is expected because both styles share similar colour palettes, painting techniques, and historical influences.

Renaissance Styles
Confusions appear between Early Renaissance, High Renaissance, and Northern Renaissance styles. These artistic movements share many visual conventions, religious subjects, and compositional techniques, making them difficult to separate consistently.

Romanticism vs Realism
Several paintings from Romanticism and Realism are confused by the model. Although these movements are historically distinct, they often depict similar subjects such as landscapes, everyday life, and human figures, making visual separation difficult.

Implications

The style classification errors are often artistically meaningful.

The model generally confuses styles that art historians would also consider visually related. Rather than making random mistakes, the network tends to group together artistic movements that share similar techniques, colour palettes, subjects, or historical influences.

This suggests that the model has learned relevant artistic representations, even when it predicts the wrong style.

### Genre Confusion Matrix

![Genre Confusion Matrix](../results/figures/resnet_confusion_test.png)

### Style Confusion Matrix

![Style Confusion Matrix](../results/figures/resnet_style_confusion_test.png)

7. Lessons Learned and Deployment Recommendation

Lessons Learned
This project demonstrates that monitoring training metrics is essential for understanding model behaviour.

Several important observations emerged:

Training accuracy alone is not a reliable metric. Models can reach very high training accuracy while performing poorly on unseen data.

Regularization plays a critical role. The logistic regression baseline showed severe overfitting when regularization was weak.

Early stopping is valuable. The ResNet models reached their best validation performance early during training, while continuing to improve only on the training set.

Style classification is significantly more challenging than genre classification. Artistic styles often share visual characteristics, making the distinction difficult even for a powerful convolutional neural network.

Deployment Recommendation
The museum requires a model that balances accuracy and computational cost.

The logistic regression baseline is extremely fast and lightweight, but its classification accuracy remains limited.

The fine-tuned ResNet18 achieves substantially higher accuracy:

Genre classification: approximately 58.6% test accuracy
Style classification: approximately 36.6% test accuracy
Although the ResNet model is larger and requires more computation, inference remains fast enough for deployment on a museum kiosk.

Recommendation:

Deploy the fine-tuned ResNet18 model in production. The increase in accuracy justifies the additional computational cost and provides a significantly better visitor experience.